# IR Agent — LangGraph Multi-Agent Design

This document explains how `services/ir-agent` runs an incident-response
investigation as a four-role LangGraph pipeline, and *why* it is built the way
it is. The companion spec is `docs/superpowers/specs/2026-06-28-ir-agent-langgraph-design.md`.

## Overview

The service exposes `POST /investigate`, which takes a synthetic security
incident and streams back a structured incident-response report over SSE. Under
the hood a `StateGraph` threads a typed `IRState` through four role-separated
nodes:

```
triage -> investigate -> validate -> (investigate | report) -> END
```

- **triage** classifies severity/category/confidence (cheap model).
- **investigate** runs a bounded tool loop over fixture-backed evidence, then
  emits grounded `Findings` (strong model).
- **validate** adversarially checks every claim is grounded in retrieved
  evidence and decides whether to loop back (strong model).
- **report** writes the final `IRReport` (mid-tier model).

Output is validated **twice**: each node is forced through Pydantic structured
output, and the validator independently re-checks grounding before the report
is allowed to be written.

## Architecture Context

The IR agent sits beside the other Python services (`chat`, `debug`, `eval`,
`rag-triage`) and reuses the shared package for cross-cutting concerns:
`shared.auth` (JWT dependency), `shared.host_validation`, `shared.logging`
(structlog), and `shared.tracing` (OpenTelemetry). It is a FastAPI app behind
the same gateway, with Prometheus metrics on `/metrics` and a `/health` probe.

Unlike `chat`/`debug`, which talk to a pluggable local LLM through the
`shared/llm` factory (Ollama/OpenAI/Anthropic), the IR agent depends on
`langchain-anthropic` + `langgraph` directly. That decision is explained below.

Evidence is **not** retrieved from a live SIEM/EDR — it is read from bundled
JSON fixtures (`services/ir-agent/fixtures/`). This keeps the whole graph
deterministic and lets every test run without network or external services.

## Package Introductions

### `langgraph`
A library for building stateful, multi-actor LLM applications as a directed
graph. We use `StateGraph` to declare nodes and edges, `add_conditional_edges`
for the validate→(investigate|report) branch, and `.compile()` to get a
runnable with `.invoke()`/`.stream()`.

*Why over alternatives?* A hand-rolled `while` loop would also work, but
LangGraph gives us a declared topology (easy to reason about and to diagram), a
single typed state object reducer, and `.stream()` for free — which is exactly
what the SSE endpoint needs. We deliberately chose a **linear pipeline with a
bounded validation loop** rather than a *supervisor* agent that dynamically
routes among sub-agents: the IR workflow has a known, fixed shape (triage →
investigate → validate → report), so a supervisor's extra LLM routing call adds
cost and nondeterminism with no benefit. The only dynamic decision is "is this
grounded enough, or investigate again?", which a single conditional edge models
cleanly.

### `langchain-core`
Provides the message types (`SystemMessage`/`HumanMessage`/`ToolMessage`/
`AIMessage`), the `@tool` decorator, and `with_structured_output()` /
`bind_tools()` on chat models. These are the seams our tests mock.

### `langchain-anthropic`
The `ChatAnthropic` chat-model implementation. We build one per role with a
different model id (see tiering below). `with_structured_output(Schema)` uses
Anthropic tool-calling under the hood to coerce output into a Pydantic model.

### `pydantic` / `pydantic-settings`
Every node emits a Pydantic model, never free text — the graph state is
type-checked end to end. `pydantic-settings` loads per-role model ids and loop
caps from `IR_`-prefixed env vars.

### `sse-starlette`
`EventSourceResponse` turns the graph's `.stream()` output into Server-Sent
Events, so the client sees `triage`, `evidence`, `findings`, `verdict`,
`report`, and `done` events as they happen rather than waiting for the full run.

## Decision: per-role model tiering

Each role is backed by its own `ChatAnthropic` instance, chosen for the
difficulty of that role's job:

| Role | Model | Why |
|------|-------|-----|
| triage | `claude-haiku-4-5` | Short classification from one incident blob — cheap + fast is enough. |
| investigate | `claude-opus-4-8` | Multi-step tool reasoning and hypothesis forming — the hardest job. |
| validate | `claude-opus-4-8` | Adversarial grounding check — must be at least as strong as the investigator it audits. |
| report | `claude-sonnet-4-6` | Structured write-up from already-validated facts — mid-tier suffices. |

Keeping the mapping in one place (`app/roles.py`) makes the cost/quality
trade-off explicit and easy to tune. Using one expensive model for every node
would burn budget on triage and report; using one cheap model everywhere would
fail at investigation and validation.

## Decision: two-layer output validation

LLM output is validated at two independent layers:

1. **Schema layer (per node).** Every node calls
   `model.with_structured_output(Schema)`, so a node can only return a value
   that parses into its Pydantic contract (`TriageResult`, `Findings`,
   `ValidationVerdict`, `IRReport`). Malformed output is retried by the model
   layer, not silently passed downstream.
2. **Grounding layer (validate node).** Structure does not imply truth. A
   well-formed `Findings` can still cite evidence that does not exist or make a
   claim no evidence supports. The validator is given the findings **and the
   exact evidence catalog** and returns a `ValidationVerdict`; if it is not
   grounded (and we are under the retry cap) the graph routes back to
   investigate with the named gaps.

The loop is **bounded** by `max_investigate_attempts` so an ungrounded answer
cannot spin forever — after the cap we proceed to report with the best
available findings (and the verdict is still attached for honesty).

## Decision: `langchain-anthropic`/`langgraph` directly, not `shared/llm`

The existing `shared/llm` factory abstracts a *single* chat/embedding provider
for the RAG services and targets Ollama-first local inference. The IR agent
needs things that abstraction does not express:

- **Per-role models in one process** (Haiku + Opus + Sonnet simultaneously).
- **Tool-calling loops** and **structured output** as first-class operations.
- **Graph orchestration** with a typed shared state and streaming.

Bending `shared/llm` to cover all of this would turn a deliberately small
abstraction into a leaky one. Depending on `langchain-anthropic`/`langgraph`
directly keeps the IR agent's needs local to the IR agent and leaves the shared
factory simple for the services that actually want provider-pluggability.

## Go/TS Comparison

Mapping the patterns to languages already in the toolbox:

| Concept | Go / TS | Python (here) |
|---------|---------|---------------|
| Typed shared state | a `struct` threaded through handlers | `IRState` `TypedDict` |
| Node = pure-ish step | `func(State) (State, error)` | `Callable[[IRState], dict]` |
| Schema-validated boundary | `json.Unmarshal` into a struct / zod parse | `with_structured_output(Model)` |
| Dependency injection for tests | interface + fake impl | `builder=` param + `FakeChatModel` |
| Conditional routing | `switch` on a result | `add_conditional_edges` + router fn |
| Streaming response | `http.Flusher` / `ReadableStream` | `EventSourceResponse` over `.stream()` |

## Build It

The cells below rebuild the design in miniature. They import the real service
package and drive the graph with **fake models** — no API key or network
needed. Run them with the `ir-agent/.venv` kernel.

In [ ]:
import sys
from pathlib import Path

# Make `app` importable: the notebook lives in docs/adr/ir-agent/, the package
# lives in services/ir-agent/app — add services/ir-agent to sys.path.
REPO = Path.cwd().resolve()
while not (REPO / "services").is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "services" / "ir-agent"))
print("package root:", REPO / "services" / "ir-agent")

### Step 1: The contracts are the API between nodes

Because every node returns a Pydantic model, a downstream node never has to
parse free text — it reads typed fields. Constructing one shows the shape.

In [ ]:
from app.models import Findings, TriageResult

triage = TriageResult(severity="high", category="phishing", confidence=0.9,
                      rationale="login from RO 3m after a credential-harvest click")
print(triage.model_dump_json(indent=2))

### Step 2: Evidence tools are deterministic and incident-scoped

`build_tools(incident_id)` closes four `@tool` functions over one incident's
fixtures, so the investigator's tool calls always return the same evidence.

> **Pitfall:** tools return *strings* (what the model reads). The investigate
> node is responsible for wrapping each result into an `EvidenceItem` with a
> stable id like `search_alerts-0` so the validator can check citations.

In [ ]:
from app.tools import build_tools

tools = {t.name: t for t in build_tools("INC-PHISH-001")}
print(sorted(tools))
print(tools["search_alerts"].invoke({"query": "login"}))

### Step 3: A node is a closure over its model

`make_triage_node(model)` returns the actual graph node. Passing a fake model
is how every node test avoids the API — the node code is identical in prod.

In [ ]:
from app.models import Incident
from app.nodes.triage import make_triage_node


class _FakeStructured:
    def __init__(self, payload):
        self._payload = payload

    def invoke(self, _messages):
        return self._payload


class _FakeModel:
    """Minimal stand-in: only the methods the nodes actually call."""

    def __init__(self, structured):
        self._structured = structured

    def with_structured_output(self, _schema):
        return _FakeStructured(self._structured)


node = make_triage_node(_FakeModel(triage))
incident = Incident(id="INC-PHISH-001", source="email-gw", title="phish then login")
print(node({"incident": incident})["triage"].severity)

### Step 4: The graph wires nodes and the bounded loop

`build_graph` adds the four nodes, the fixed edges, and the one conditional
edge. `route_after_validate` is the only branch: loop back to investigate while
ungrounded and under the cap, otherwise go to report.

In [ ]:
from langchain_core.messages import AIMessage

from app.graph import build_graph
from app.models import IRReport, ValidationVerdict


class _ToolRunnable:
    def __init__(self, scripted):
        self._scripted = list(scripted)

    def invoke(self, _messages):
        return self._scripted.pop(0) if self._scripted else AIMessage(content="done")


class _FullFake(_FakeModel):
    def __init__(self, structured, tool_script=None):
        super().__init__(structured)
        self._tool_script = tool_script or []

    def bind_tools(self, _tools):
        return _ToolRunnable(self._tool_script)


models = {
    "triage": _FullFake(triage),
    "investigate": _FullFake(
        Findings(summary="creds phished", hypothesis="account takeover",
                 evidence_refs=["search_alerts-0"]),
        tool_script=[AIMessage(content="", tool_calls=[{"name": "search_alerts",
                     "args": {"query": "login"}, "id": "c1", "type": "tool_call"}]),
                     AIMessage(content="done")]),
    "validate": _FullFake(ValidationVerdict(grounded=True,
                          needs_more_investigation=False)),
    "report": _FullFake(IRReport(executive_summary="account takeover via phish",
                        severity="high", confidence=0.85)),
}

graph = build_graph(models, max_tool_steps=4, max_attempts=2)
out = graph.invoke({"incident": incident, "evidence": [], "investigate_attempts": 0})
print("report severity:", out["report"].severity)
print("investigate passes:", out["investigate_attempts"])

## Experiment

Three tweaks that reveal why the chosen design holds:

1. **Make the validator always ungrounded.** Replace the `validate` fake with
   `ValidationVerdict(grounded=False, needs_more_investigation=True)` and run
   again. Observe `investigate_attempts` stops at `max_attempts` — proof the
   loop is bounded, not infinite.
2. **Lower `max_attempts` to 1.** The graph reports after a single
   investigation even when ungrounded. This is the cost/quality knob: more
   attempts = more grounded but more tokens.
3. **Swap the per-role models.** In `app/roles.py`, point `investigate` at the
   Haiku id and watch (with a live key) grounding quality drop — the tiering
   table is not arbitrary.

## Check Your Understanding

1. Why validate grounding in a separate node instead of just trusting the
   investigator's own `with_structured_output(Findings)`? What failure does the
   schema layer *not* catch?
2. We chose a linear-pipeline-with-loop over a supervisor agent. Under what
   change to the requirements would a supervisor start to pay for itself?
3. The validate→investigate edge is bounded by `max_attempts`. What would break
   if it were unbounded, and how does that compare to a retry budget you'd set
   on a Go HTTP client?